In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.font_manager as fm
import seaborn as sns

In [ ]:
# 시각화를 위한 폰트 설정
font_list = fm.findSystemFonts(fontpaths=None, fontext='ttf')
font_list[:10]  # 일부만 출력
font_path='C:\\Windows\\Fonts'
mpl.rcParams['font.family'] = 'Malgun Gothic'

#### 데이터 불러오기

In [ ]:
total_pop = pd.read_excel('../P1_Data/02.Pretreatment/pre_total_pop.xlsx')
up65_pop = pd.read_excel('../P1_Data/02.Pretreatment/pre_up65_pop.xlsx')

In [ ]:
total_pop

In [ ]:
up65_pop

In [ ]:
mapping = pd.merge(total_pop,up65_pop, on='행정기관')
mapping

## 수정필요 (소수점 삭제 및 비율순으로 정렬)

In [ ]:
use_col = mapping[['행정기관','총인구', '65세이상_합계']]

# 비율 계산
df_ratio = use_col.apply(lambda x: (x['65세이상_합계'] / x['총인구']) * 100, axis=1)
df_ratio = df_ratio.apply(lambda x: round(x,1))

# 새로운 데이터프레임 생성
result = pd.concat([use_col, df_ratio], axis=1)
result.columns = ['행정기관', '총인구','65세이상_합계', '65세이상 고령자 비율']
# 비율값을 구한 결과 데이터프레임을 내림차순으로 정렬하여 저장합니다.
result_sorted = result.sort_values('65세이상 고령자 비율', ascending=False)

result_sorted

In [ ]:
# 공주시 데이터를 따로 저장합니다.
gongju = result_sorted.loc[result['행정기관'] == '공주시']

# 지역명이 읍,면으로 끝나는 데이터를 저장합니다.
eup_myeon = result_sorted.loc[result_sorted['행정기관'].str.endswith('읍') | result_sorted['행정기관'].str.endswith('면')]

# 지역명이 동으로 끝나는 데이터를 저장합니다.
dong = result_sorted.loc[result_sorted['행정기관'].str.endswith('동')]

In [ ]:
eup_myeon

In [ ]:
dong

In [ ]:
result_sigol = pd.concat([gongju, eup_myeon])
result_sigol

In [ ]:
result_dosi = pd.concat([gongju, dong])
result_dosi

#### 저장

In [ ]:
# 소수점 이하 1자리를 유지하면서 결과 저장
result_sigol.to_excel('../P3_Result/01.CSV/result_sigol_up65.xlsx', index=False, float_format='%.1f')
# 소수점 이하 1자리를 유지하면서 결과 저장
result_dosi.to_excel('../P3_Result/01.CSV/result_dosi_up65.xlsx', index=False, float_format='%.1f')

#### 시각화

In [ ]:
df = result.iloc[1:] #공주시 행 제외

ax = df.plot(x='행정기관',y=['총인구','65세이상_합계'],kind='bar',figsize=(10,6),title='공주시 65세이상 고령자 비율',ylabel= '인구수',width=0.7)
plt.xticks(rotation=45)

# 특정컬럼 값들을 가져와서 막대 위에 표시
for i, row in df.iterrows():
    v = row['65세이상 고령자 비율']
    a = row['65세이상_합계']
    ax.text(i-1, a+100, v, color='black', fontsize=10,ha='center') #f'{v:.1f}%'

plt.show()

In [ ]:
df = result.iloc[1:] #공주시 행 제외

ax = df.plot(x='행정기관', y='총인구', kind='bar', figsize=(10, 6), title='공주시 65세이상 고령자 비율', ylabel='인구수', width=0.7)
ay = df.plot(x='행정기관', y='65세이상_합계', kind='line', ax=ax, secondary_y=True)

plt.xticks(rotation=45)

# 특정컬럼 값들을 가져와서 막대 위에 표시
for i, row in df.iterrows():
    v = row['65세이상 고령자 비율']
    a = row['65세이상_합계']
    ay.text(i-1, a,f'{v:.1f}%', color='black', fontsize=10, ha='center')

plt.show()

In [ ]:
ax = df.plot(x='행정기관',y=['65세이상_합계'],kind='bar',figsize=(10,6),title='공주시 65세 이상 고령자 비율',ylabel= '세대수',width=0.7,color='orange')
plt.xticks(rotation=45)

# 특정컬럼 값들을 가져와서 막대 위에 표시
for i, row in df.iterrows():
    v = row['65세이상 고령자 비율']
    a = row['65세이상_합계']
    ax.text(i-1, a+50, f'{v:.1f}%', color='black', fontsize=10,ha='center')

plt.show()

In [ ]:
# 파이차트로그리기
df_pie =df.copy()
df_pie.set_index('행정기관', inplace=True)
# 파이차트 그리기
df_pie['65세이상 고령자 비율'].plot(kind='pie', autopct='%1.1f%%', startangle=90, counterclock=False,labeldistance=1.2)
plt.axis('equal')
plt.legend(labels=df_pie.index, loc='upper right',bbox_to_anchor=(1.2, 1))
plt.title('공주시 읍면동별 65세이상 고령자 비율',y=1.1)
plt.show()